In [0]:
dbutils.widgets.text("Incremental_flag",'0')

In [0]:
Incremental_flag= dbutils.widgets.get("Incremental_flag")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import * 

##Getting silver Data

In [0]:

df = spark.read.format('parquet').load('abfss://silver@retailstgaccpd.dfs.core.windows.net/cleaned_data/')
        

## fetching relative columns

In [0]:
df_src = spark.sql(''' 
SELECT DISTINCT (Date_ID) as Date_ID from parquet.`abfss://silver@retailstgaccpd.dfs.core.windows.net/cleaned_data`
''')

#Getting schema of source df with empty

### dim_model sink initial and incremental load

In [0]:
if spark.catalog.tableExists('cars_data_catalog.gold.dim_date'):
    df_sink= spark.sql('''
        SELECT dim_date_key, Date_ID from cars_data_catalog.gold.dim_date
        ''')
else:
    df_sink= spark.sql('''
    SELECT 1 as dim_date_key, Date_ID from parquet.`abfss://silver@retailstgaccpd.dfs.core.windows.net/cleaned_data` 
    where 1=0
    ''')


### Filtering new records vs new records

In [0]:
df_filtered = df_src.join(df_sink, df_src.Date_ID==df_sink.Date_ID,how='left').select(df_src['Date_ID'], df_sink['dim_date_key'])

### df_filter_old

In [0]:
df_filter_old = df_filtered.filter(col('dim_date_key').isNotNull())

In [0]:
df_filter_new = df_filtered.filter(col('dim_date_key').isNull()).select(df_filtered['Date_ID'])


### Create Surrogate Key

**get max value for surrogate key **

In [0]:
if (Incremental_flag== '0'):
    max_value=1
else:
    max_value_df = spark.sql("SELECT MAX(dim_date_key)  FROM cars_data_catalog.gold.dim_date")
    max_value = max_value_df.collect()[0][0]+1

**create a new surrogate key and add max value in it**

In [0]:
df_filter_new = df_filter_new.withColumn('dim_date_key', max_value + monotonically_increasing_id())

### Create Final df = filter_new + filter_old

In [0]:
df_final = df_filter_new.union(df_filter_old)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_data_catalog.gold.dim_date'):
    delta_tbl= DeltaTable.forPath(spark, "abfss://gold@retailstgaccpd.dfs.core.windows.net/dim_date")

    delta_tbl.alias("trg").merge(df_final.alias("src"), "trg.dim_date_key = src.dim_date_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_final.write.format("delta").\
    mode("overwrite")\
    .option("path", "abfss://gold@retailstgaccpd.dfs.core.windows.net/dim_date")\
    .saveAsTable('cars_data_catalog.gold.dim_date')

In [0]:
%sql
select * from cars_data_catalog.gold.dim_date;

Date_ID,dim_date_key
DT00029,1
DT00030,2
DT00039,3
DT00078,4
DT00116,5
DT00120,6
DT00124,7
DT00136,8
DT00140,9
DT00164,10
